# 06b · Training Detectors `[kaggle]` (smoke-tests locally)

**Goal:** train detectors on new data, in the two styles you'll meet at work: a **hand-written PyTorch loop** around torchvision's Faster R-CNN, and the **Ultralytics YOLO** workflow, including converting annotations and labeling your own images.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | What changes when you fine-tune a detector; box-aware augmentation |
| 🔨 Project 1 | Fine-tune Faster R-CNN on Penn-Fudan pedestrians with your own training loop |
| 🔨 Project 2 | Convert VOC annotations to YOLO format from scratch, then train YOLO11 on VOC |
| 🏷 Your data | Labeling workflow (Label Studio / Roboflow / CVAT) → YOLO format → same training code |
| 📊 Evaluate | Before vs. after fine-tuning with your `cvlib.detection` evaluator, cross-checked against Ultralytics |
| 🎥 Live | `webcam_detect.py`: your trained model on your webcam |
| 🧪 Experiment | The flipped-boxes bug, mosaic augmentation, how many images you need |

### Run modes
- **Kaggle (GPU T4):** `FULL` mode, ~30–40 min. Attach your `cvlib` dataset, turn on Internet.
- **Your Mac:** `SMOKE` mode, tiny subsets and 1 epoch (~5 min). Detection training at full image size runs at only ~0.7 images/s on Apple GPUs, so the real runs belong on Kaggle.

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import os
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
from PIL import Image
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights,
                                          fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from cvlib.datasets import pennfudan, voc2007_detection, VOC_CLASSES, VOC_TO_COCO
from cvlib.detection import evaluate_detections, error_breakdown, box_convert
from cvlib.data import model_path
from cvlib.viz import show

os.environ["YOLO_VERBOSE"] = "False"
device = env.device
FULL = env.platform == "kaggle" or os.environ.get("CV_FULL") == "1"
CFG = dict(
    frcnn="resnet50_v2" if FULL else "mobilenet",
    frcnn_epochs=10 if FULL else 1,
    frcnn_train=None if FULL else 16,          # cap on training images in smoke mode
    frcnn_min_size=800 if FULL else 320,       # torchvision resizes the short side to this
    yolo_epochs=30 if FULL else 1,
    yolo_imgsz=640 if FULL else 320,
    yolo_train=None if FULL else 64,
    yolo_val=None if FULL else 32,
)
print("MODE:", "FULL" if FULL else "SMOKE (numbers are not meaningful)", CFG)

---
# 🧠 1 · Intuition

### What fine-tuning a detector changes
A COCO-pretrained detector already knows "objectness" and 80 classes. To adapt it:
- **New classes:** replace the classification head (and the class-specific box regressor) with one sized for *your* classes. The backbone, FPN and RPN keep their pretrained weights.
- **Same classes, new domain** (Penn-Fudan pedestrians are COCO "persons"): fine-tuning still helps, by adapting to *your* camera angles, object sizes and, importantly, *your annotators' box style* (do boxes include the umbrella? the shadow?).

### Augmentation must move the boxes too
Flip, crop or scale the image and the boxes have to be transformed the same way. Get this wrong and the model learns from silently corrupted labels (Experiment 1 shows how badly). Libraries like Ultralytics and albumentations do it for you; in a hand-written loop it's your job.

### Detection training loops differ from classification
torchvision detection models compute their **own losses** in `train()` mode (they need the targets internally to match anchors/proposals). You get a dict: RPN objectness + RPN box regression + classifier + box regression losses. Sum them and backprop. Images in a batch can have **different sizes** and **different numbers of boxes**, so batches are lists rather than single tensors.

---
# 🔨 2 · Project 1: fine-tune Faster R-CNN on Penn-Fudan

In [ ]:
items = pennfudan()
rng = np.random.default_rng(0)
perm = rng.permutation(len(items))
train_items = [items[i] for i in perm[:120]]
test_items = [items[i] for i in perm[120:]]
if CFG["frcnn_train"]:
    train_items = train_items[:CFG["frcnn_train"]]
print(f"train {len(train_items)} images | test {len(test_items)} images | {sum(len(i['boxes']) for i in test_items)} test pedestrians")

def load(it):
    return torchvision.transforms.functional.to_tensor(Image.open(it["path"]).convert("RGB"))

vis = cv2.cvtColor(np.array(Image.open(train_items[0]["path"]).convert("RGB")), cv2.COLOR_RGB2BGR)
for b in train_items[0]["boxes"].astype(int):
    cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), (0, 200, 0), 2)
show(vis, "a Penn-Fudan image with its boxes", size=5)

### 2.1 · Box-aware augmentation: horizontal flip

In [ ]:
def hflip(img, boxes):
    """Flip an image tensor (C, H, W) left-right, and its xyxy boxes with it."""
    W = img.shape[-1]
    boxes = boxes.clone()
    boxes[:, [0, 2]] = W - boxes[:, [2, 0]]                       # new x1 = W − old x2, new x2 = W − old x1
    return img.flip(-1), boxes

class DetectionData(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        img = load(it)
        boxes = torch.tensor(it["boxes"], dtype=torch.float32)
        if self.train and torch.rand(1) < 0.5:
            img, boxes = hflip(img, boxes)
        target = {"boxes": boxes, "labels": torch.ones(len(boxes), dtype=torch.int64)}   # class 1 = person (0 = background)
        return img, target

collate = lambda batch: tuple(zip(*batch))                         # keep lists: images differ in size
train_dl = torch.utils.data.DataLoader(DetectionData(train_items, True), batch_size=4, shuffle=True, collate_fn=collate)

### 2.2 · The model: swap the head for 2 classes (background + person)

In [ ]:
def build_frcnn(n_classes=2):
    if CFG["frcnn"] == "resnet50_v2":
        m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT, min_size=CFG["frcnn_min_size"])
    else:
        m = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT, min_size=CFG["frcnn_min_size"])
    in_feat = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, n_classes)  # new: class scores + per-class box deltas
    return m

model = build_frcnn().to(device)
print(model.roi_heads.box_predictor)

### 2.3 · Before training: how good is the COCO model already?

The COCO detector already knows "person". Score its person detections on the test set, so we know what fine-tuning actually adds.

In [ ]:
@torch.no_grad()
def predict_frcnn(m, items, keep_label=1):
    m.eval(); m.roi_heads.score_thresh = 0.001
    preds = []
    for it in items:
        d = m([load(it).to(device)])[0]
        sel = (d["labels"] == keep_label).cpu().numpy()
        preds.append(dict(boxes=d["boxes"].cpu().numpy()[sel], scores=d["scores"].cpu().numpy()[sel], labels=np.zeros(sel.sum(), int)))
    return preds

test_gts = [dict(boxes=it["boxes"], labels=np.zeros(len(it["boxes"]), int)) for it in test_items]
coco_model = (fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT, min_size=CFG["frcnn_min_size"])
              if CFG["frcnn"] == "resnet50_v2" else
              fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT, min_size=CFG["frcnn_min_size"])).to(device)
before = evaluate_detections(test_gts, predict_frcnn(coco_model, test_items, keep_label=1), n_classes=1)   # COCO label 1 = person
print(f"COCO model, no fine-tuning: AP50 {before['AP50']:.3f} | mAP {before['mAP']:.3f}")
del coco_model

### 2.4 · The training loop

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG["frcnn_epochs"])
history = []
for epoch in range(1, CFG["frcnn_epochs"] + 1):
    model.train(); t0 = time.perf_counter(); sums = {}
    for imgs, targets in train_dl:
        imgs = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        losses = model(imgs, targets)                               # train mode → dict of losses
        loss = sum(losses.values())
        opt.zero_grad(); loss.backward(); opt.step()
        for k, v in losses.items():
            sums[k] = sums.get(k, 0) + v.item() / len(train_dl)
    sched.step()
    history.append(sums)
    print(f"epoch {epoch:2d} | " + " | ".join(f"{k.replace('loss_', '')} {v:.3f}" for k, v in sums.items()) + f" | {time.perf_counter() - t0:.0f}s")

after = evaluate_detections(test_gts, predict_frcnn(model, test_items, keep_label=1), n_classes=1)
print(f"\nCOCO model, no fine-tuning : AP50 {before['AP50']:.3f} | mAP {before['mAP']:.3f}")
print(f"fine-tuned on Penn-Fudan   : AP50 {after['AP50']:.3f} | mAP {after['mAP']:.3f}")

The four losses tell you which part is learning: `objectness` and `rpn_box_reg` are stage 1 (does the RPN find people?), `classifier` and `box_reg` are stage 2.

On Kaggle (`FULL`), expect AP50 to be high both before and after: COCO already detects people well. The gain shows mostly in **mAP** (strict IoU), as the model learns Penn-Fudan's exact box style. In smoke mode the model trains for one tiny epoch, so "after" can be *worse* than "before". That's expected: the new head starts random.

In [ ]:
model.eval()
fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
for ax, it in zip(axes, test_items[:4]):
    with torch.no_grad():
        d = model([load(it).to(device)])[0]
    v = np.array(Image.open(it["path"]).convert("RGB"))
    for b in it["boxes"].astype(int):
        cv2.rectangle(v, tuple(b[:2]), tuple(b[2:]), (0, 200, 0), 2)
    for b, s in zip(d["boxes"].cpu().numpy().astype(int), d["scores"].cpu().numpy()):
        if s > 0.5:
            cv2.rectangle(v, tuple(b[:2]), tuple(b[2:]), (255, 0, 0), 2)
    ax.imshow(v); ax.axis("off")
plt.suptitle("green = ground truth | red = fine-tuned predictions (score > 0.5)"); plt.tight_layout(); plt.show()

---
# 🔨 3 · Project 2: train YOLO11 on PASCAL VOC

### 3.1 · Convert VOC XML to YOLO format, from scratch

Ultralytics expects:
```
dataset/
  images/train/xxx.jpg     labels/train/xxx.txt
  images/val/yyy.jpg       labels/val/yyy.txt
data.yaml                  (paths + class names)
```
Each label file has **one line per object**: `class_id cx cy w h`, all normalized to 0–1 by the image size. Images without objects get an empty file (they still teach the model what "background" looks like).

In [ ]:
def voc_to_yolo(items, split, root, skip_difficult=True):
    (root / "images" / split).mkdir(parents=True, exist_ok=True)
    (root / "labels" / split).mkdir(parents=True, exist_ok=True)
    for it in items:
        dst = root / "images" / split / it["path"].name
        if not dst.exists():
            try:
                os.symlink(it["path"], dst)                          # no copy needed
            except OSError:
                shutil.copy(it["path"], dst)
        keep = ~it["difficult"] if skip_difficult else np.ones(len(it["labels"]), bool)
        cxcywh = box_convert(it["boxes"][keep], "xyxy", "cxcywh") / np.array([it["width"], it["height"]] * 2)
        lines = [f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, (x, y, w, h) in zip(it["labels"][keep], cxcywh)]
        (root / "labels" / split / f"{it['path'].stem}.txt").write_text("\n".join(lines))

voc_train, voc_val = voc2007_detection("train"), voc2007_detection("val")
if CFG["yolo_train"]:
    voc_train, voc_val = voc_train[:CFG["yolo_train"]], voc_val[:CFG["yolo_val"]]
yolo_root = env.data_dir / ("voc_yolo" if FULL else "voc_yolo_smoke")
voc_to_yolo(voc_train, "train", yolo_root); voc_to_yolo(voc_val, "val", yolo_root)
data_yaml = yolo_root / "data.yaml"
data_yaml.write_text(f"path: {yolo_root}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(VOC_CLASSES)))
print(data_yaml.read_text()[:200], "...")
print("example label file:\n" + (yolo_root / "labels" / "train" / f"{voc_train[0]['path'].stem}.txt").read_text())

In [ ]:
# Sanity check: draw a converted label file back onto its image. Always do this after any format conversion!
it = voc_train[0]
img = cv2.imread(str(it["path"])); H, W = img.shape[:2]
for line in (yolo_root / "labels" / "train" / f"{it['path'].stem}.txt").read_text().splitlines():
    c, cx, cy, w, h = line.split()
    x1, y1, x2, y2 = box_convert(np.array([[float(cx) * W, float(cy) * H, float(w) * W, float(h) * H]]), "cxcywh", "xyxy")[0].astype(int)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(img, VOC_CLASSES[int(c)], (x1, y1 + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 2)
show(img, "YOLO label file drawn back onto its image", size=5)

### 3.2 · Train

Start from COCO-pretrained YOLO11n weights; Ultralytics replaces the 80-class head with a 20-class one automatically. It brings its own augmentation pipeline (**mosaic**, scale, color, flips: all box-aware), warmup, cosine/linear LR schedule, EMA weights and validation.

In [ ]:
from ultralytics import YOLO

def train_yolo(data_yaml, epochs, imgsz, name, base="yolo11n.pt", **kw):
    m = YOLO(str(model_path(base)))
    m.train(data=str(data_yaml), epochs=epochs, imgsz=imgsz, batch=16 if FULL else 8, device=device,
            project=str(env.out_dir / "yolo_runs"), name=name, exist_ok=True, plots=True, verbose=False, workers=2 if FULL else 0,
            deterministic=False, **kw)                                   # deterministic mode only adds warnings on Apple GPUs
    return m

t0 = time.perf_counter()
yolo_voc = train_yolo(data_yaml, CFG["yolo_epochs"], CFG["yolo_imgsz"], name="voc_" + ("full" if FULL else "smoke"))
run_dir = env.out_dir / "yolo_runs" / ("voc_" + ("full" if FULL else "smoke"))
print(f"trained in {(time.perf_counter() - t0) / 60:.1f} min → {run_dir}")

In [ ]:
res = pd.read_csv(run_dir / "results.csv")
res.columns = [c.strip() for c in res.columns]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for k in ("train/box_loss", "train/cls_loss", "train/dfl_loss"):
    axes[0].plot(res["epoch"], res[k], marker="o", label=k.split("/")[1])
axes[0].set_title("training losses"); axes[0].legend()
axes[1].plot(res["epoch"], res["metrics/mAP50(B)"], marker="o", label="mAP50"); axes[1].plot(res["epoch"], res["metrics/mAP50-95(B)"], marker="o", label="mAP50-95")
axes[1].set_title("validation mAP"); axes[1].legend()
axes[2].plot(res["epoch"], res["metrics/precision(B)"], marker="o", label="precision"); axes[2].plot(res["epoch"], res["metrics/recall(B)"], marker="o", label="recall")
axes[2].set_title("validation precision / recall"); axes[2].legend()
for a in axes: a.set_xlabel("epoch"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

`box_loss` (IoU-based box quality), `cls_loss` (classification) and `dfl_loss` (Distribution Focal Loss: YOLOv8+ predicts each box edge as a small probability distribution instead of one number, which sharpens localization).

---
# 📊 4 · Evaluate: before vs. after, and cross-checking Ultralytics

Score three models on the same VOC validation images with **your** evaluator (difficult boxes ignored):
1. COCO-pretrained YOLO11n, mapped to VOC classes (the 06a approach, no training)
2. YOLO11n fine-tuned on VOC
And compare your number with what Ultralytics reports for model 2.

In [ ]:
val_gts = [dict(boxes=it["boxes"], labels=it["labels"], ignore=it["difficult"]) for it in voc_val]

def yolo_preds(m, items, imgsz, class_map=None):
    out = []
    for it in items:
        r = m.predict(str(it["path"]), imgsz=imgsz, conf=0.001, device=device, verbose=False, max_det=100)[0]
        lab = r.boxes.cls.cpu().numpy().astype(int)
        keep = np.isin(lab, list(class_map)) if class_map else np.ones(len(lab), bool)
        out.append(dict(boxes=r.boxes.xyxy.cpu().numpy()[keep], scores=r.boxes.conf.cpu().numpy()[keep],
                        labels=np.array([class_map[l] if class_map else l for l in lab[keep]], dtype=np.int64)))
    return out

coco_yolo = YOLO(str(model_path("yolo11n.pt")))
cmap = {k: VOC_CLASSES.index(v) for k, n in coco_yolo.names.items() for v, c in VOC_TO_COCO.items() if n == c}
r_before = evaluate_detections(val_gts, yolo_preds(coco_yolo, voc_val, CFG["yolo_imgsz"], cmap), n_classes=20)
preds_after = yolo_preds(yolo_voc, voc_val, CFG["yolo_imgsz"])
r_after = evaluate_detections(val_gts, preds_after, n_classes=20)
ultra = yolo_voc.val(data=str(data_yaml), imgsz=CFG["yolo_imgsz"], device=device, verbose=False, plots=False, workers=0)

print(f"{'':<34}{'mAP50':>7}{'mAP50-95':>10}")
print(f"{'COCO YOLO11n, no training (ours)':<34}{r_before['AP50']:>7.3f}{r_before['mAP']:>10.3f}")
print(f"{'YOLO11n fine-tuned on VOC (ours)':<34}{r_after['AP50']:>7.3f}{r_after['mAP']:>10.3f}")
print(f"{'YOLO11n fine-tuned (Ultralytics)':<34}{ultra.box.map50:>7.3f}{ultra.box.map:>10.3f}")
print("\nerror breakdown, fine-tuned:", error_breakdown(val_gts, preds_after))

Your evaluator and Ultralytics' should land close but **not identical**: Ultralytics interpolates the PR curve differently (and treats VOC's *difficult* boxes as regular boxes, while we ignore them as the VOC protocol says). Small differences in evaluation code are normal across libraries, which is exactly why you **compare models with the same evaluator**, never numbers copied from different codebases.

In `FULL` mode the fine-tuned model should clearly beat the COCO model: same architecture, but now trained on VOC's exact classes, box style and image statistics.

---
# 🏷 5 · Train on *your own* images

The skill that matters most in practice: going from "a folder of photos" to a working detector.

**1. Collect.** Photos that look like deployment: same camera, angles, lighting, backgrounds. Start with **~50–150 images per class**. Include **negative images** (scenes without your objects) to cut false alarms.

**2. Label.** Pick a tool:
| Tool | Setup | Notes |
|---|---|---|
| [Label Studio](https://labelstud.io) | `uvx label-studio` (runs locally in your browser) | free, private, exports YOLO |
| [Roboflow](https://roboflow.com) | web app | free tier, auto-splits, augmentations, exports YOLO |
| [CVAT](https://www.cvat.ai) | web app or Docker | strong for big teams and video |

Rules for consistent labels: box the **visible** part tightly; decide once whether occluded or truncated objects count; label **every** instance in every image (a missed object teaches the model that it's background).

**3. Export** in *YOLO* format, put it under `data/my_dataset/` with the folder layout from section 3.1, and write a `data.yaml` with your class names.

**4. Split by scene, not by image.** If 10 photos were taken of the same desk, they all go in train *or* all in val. Otherwise validation is just "recognize the same photo again" (leakage, module 14).

**5. Train** with the same function. The cell below runs if your dataset exists:

In [ ]:
MY_DATA = env.data_dir / "my_dataset" / "data.yaml"
if MY_DATA.exists():
    my_model = train_yolo(MY_DATA, epochs=50 if FULL else 3, imgsz=640, name="my_dataset")
    metrics = my_model.val(data=str(MY_DATA), device=device, verbose=False)
    print(f"your detector: mAP50 {metrics.box.map50:.3f} | mAP50-95 {metrics.box.map:.3f}")
    print("best weights:", env.out_dir / "yolo_runs" / "my_dataset" / "weights" / "best.pt")
else:
    print(f"No dataset at {MY_DATA} yet. Label some images, export as YOLO, then re-run this cell.")

---
# 🎥 6 · Live: your detector on your webcam

`webcam_detect.py` (in this folder) runs any YOLO weights live, with FPS shown. From the project root:

```bash
uv run python notebooks/06_detection/webcam_detect.py                                   # COCO YOLO11n
uv run python notebooks/06_detection/webcam_detect.py --weights data/outputs/yolo_runs/my_dataset/weights/best.pt
```

Weights trained on Kaggle: download `best.pt` from the notebook's output panel and pass its path.

---
# 🧪 7 · Experiments: predict first, then run

**Experiment 1: the flipped-boxes bug.** Flip the image but forget to flip the boxes. Predict the IoU between the correct and the buggy labels.

In [ ]:
# Prediction: ...
it = train_items[1]
img_t = load(it); boxes = torch.tensor(it["boxes"])
flipped_img, correct = hflip(img_t, boxes)
buggy = boxes                                                     # forgot to transform the boxes
iou = torchvision.ops.box_iou(correct, buggy).diagonal()
v = (flipped_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()
for b in correct.int().tolist(): cv2.rectangle(v, tuple(b[:2]), tuple(b[2:]), (0, 200, 0), 2)
for b in buggy.int().tolist(): cv2.rectangle(v, tuple(b[:2]), tuple(b[2:]), (255, 0, 0), 2)
show(v, "flipped image: green = correctly flipped boxes, red = buggy (unflipped) boxes", bgr=False, size=5)
print("IoU between correct and buggy label for each person:", [round(v, 2) for v in iou.tolist()])
# With p = 0.5 flipping, half of all training labels would point at the wrong place: the model learns
# blurry, symmetric guesses and mAP collapses, with no error message. Always draw augmented samples.

**Experiment 2: mosaic augmentation.** Ultralytics saves the first training batches as images. Predict what a "mosaic" sample looks like before you look.

In [ ]:
# Prediction: ...
batch_imgs = sorted(run_dir.glob("train_batch*.jpg"))
if batch_imgs:
    show(cv2.imread(str(batch_imgs[0])), f"{batch_imgs[0].name}: each tile is a mosaic of 4 training images", size=9)
# Mosaic stitches 4 images into one: more objects per batch, objects at varied scales and positions,
# and partially cut objects. It's turned off for the last epochs (close_mosaic) so the model also sees normal images.

**Experiment 3 (`FULL` only): how many images do you need?** Train on 10% of VOC train and compare. Predict the drop.

In [ ]:
# Prediction: ...
if FULL:
    small_items = voc2007_detection("train")[::10]
    small_root = env.data_dir / "voc_yolo_10pct"
    voc_to_yolo(small_items, "train", small_root); voc_to_yolo(voc_val, "val", small_root)
    (small_root / "data.yaml").write_text(data_yaml.read_text().replace(str(yolo_root), str(small_root)))
    m10 = train_yolo(small_root / "data.yaml", CFG["yolo_epochs"], CFG["yolo_imgsz"], name="voc_10pct")
    r10 = evaluate_detections(val_gts, yolo_preds(m10, voc_val, CFG["yolo_imgsz"]), n_classes=20)
    print(f"10% of training data ({len(small_items)} images): mAP50 {r10['AP50']:.3f} | 100%: {r_after['AP50']:.3f}")
else:
    print("skipped in SMOKE mode (needs a GPU): run on Kaggle")
# Pretrained detectors are surprisingly data-efficient: a few hundred well-labeled images often get you
# most of the way. Label quality and coverage of real conditions matter more than raw count.

---
# 📝 8 · Recap

**Cheat sheet**
- Fine-tuning a detector = new class head (+ box regressor), pretrained backbone/FPN/RPN, small LR, a few epochs.
- torchvision detection models return a **dict of losses** in `train()` mode; batches are **lists** (variable sizes and box counts).
- **Augmentations must transform boxes** too. Draw augmented samples to check.
- **YOLO format**: one `.txt` per image, `class cx cy w h` normalized to 0–1; empty file = no objects. Always draw converted labels back.
- Ultralytics `YOLO(weights).train(data=yaml, ...)`: mosaic and other box-aware augmentations, EMA, schedules built in.
- Compare models with **one evaluator**; different libraries' mAP numbers differ slightly.
- Own data: collect deployment-like images + negatives, label consistently, **split by scene**, start with ~100 images per class.

**Interview-style questions**
1. What do you replace when fine-tuning Faster R-CNN on new classes, and what do you keep?
2. Why do detection DataLoaders need a custom `collate_fn`?
3. Describe a silent bug in a detection augmentation pipeline and how you'd catch it.
4. Write the YOLO label line for a box from (100, 50) to (300, 250) in a 640×480 image, class 3.
5. Why might your mAP differ from the one Ultralytics prints for the same model?
6. What is mosaic augmentation, and why is it switched off at the end of training?
7. You have 2 days to build a detector for a new product. Walk through your plan.

**Next:** `07a_semantic_segmentation.ipynb`: from boxes to pixels.